# Sampler example

This notebook demonstrates how to use the `LatinHyperCubeSampler` class to sample set of parameters in a Latin Hypercube (LHC) and save the results to CSV files.

This class expects a set of ranges to select the parameters from. The ranges are defined as a dictionary where the keys are the parameter names and the values are tuples representing the minimum and maximum values for each parameter.

> The `sunbird.inference.priors` module allows the selection of such ranges from its existing priors.

In [ ]:
import pandas as pd

from acm.utils.sampler import LatinHyperCubeSampler

# Let's define the parameter ranges for the sampling
ranges = {
    "param1": (0.0, 1.0),
    "param2": (10.0, 20.0),
    "param3": (100.0, 200.0),
}

lhc = LatinHyperCubeSampler(ranges=ranges, seed=42)

We can generate a sample of parameters using the `sample` method of the `LatinHyperCubeSampler` class. This returns a pandas DataFrame containing the sampled parameters.

In [ ]:
sample = lhc.sample(150)

sample.head(5) # print the first 5 rows of the sample

We can split the sample into multiple sets using the `split` method. This method takes a list of keys and splits the sample in an equal number of rows for each key. The result is a dictionary where the keys are the provided keys and the values are the corresponding split DataFrames.

In [ ]:
keys = ["set1", "set2"]

splits = lhc.split(sample, keys=keys)

for key, df in splits.items():
    print(f"Split for {key}:")
    print(df.head(5))  # print the first 5 rows of each split

It is possible to add some common parameters to each row of a sample or split using the `add_columns` method, and providing it with a DataFrame containing the common parameters. The columns of the provided DataFrame will be added to each row of the sample or split.

In [ ]:
extra_params = {
    "set1": pd.DataFrame({"p0": [1], "p1": [2]}),
    "set2": pd.DataFrame({"p0": [3], "p1": [4]}),
} # Here, we make one set of extra parameters for each split.

for key, df in splits.items():
    splits[key] = lhc.add_columns(df, extra_params[key])
    print(f"Split for {key} with extra parameters:")
    print(splits[key].head(5))  # print the first 5 rows of each split with extra parameters

Finally, we can save the sample and splits to CSV files using the `save` method. The `order` parameter allows us to specify the order of the columns in the saved CSV files.

This method also allows for saving the splits with a filename pattern that includes the `key` format, which will be replaced by the corresponding key for each split when saving the files.

In [ ]:
order = ["p0", "p1", "param1", "param2", "param3"]

# Save the sample, not including the extra parameters, to a CSV file
lhc.save(sample, save_fn="mock_data/lhc_parameters/sample.csv", order=order[2:])

# Save the splits, including the extra parameters, to CSV files with a filename pattern that includes the key
lhc.save(splits, save_fn="mock_data/lhc_parameters/sample_{key}.csv", order=order)